# Preparing input BAMs

TxNova starts from already-aligned, coordinate-sorted, indexed BAMs. It
does not align FASTQs. This page is the recipes: STAR or HISAT2, sort /
index, `strandedness`, then `txnova preflight`.

Sections 1–6 are commands, not executed. Section 7 runs on the tiny
fixtures in `tests/fixtures/` so you can see a passing preflight and the
usual contig / aligner failures.

Already have STAR or HISAT2 BAMs? Skip to [§4](#4-sort-index-and-verify)
or jump to [§7](#7-preflight-on-the-bundled-fixtures).


In [1]:
from pathlib import Path

import pandas as pd
import yaml

import txnova
from txnova.config import load_config
from txnova.preflight import run_preflight
from txnova.samples import load_samples

print("txnova", txnova.__version__)


txnova 0.1.3


## 1. Pick an aligner

TxNova accepts **STAR** or **HISAT2** — not a mix in one run. Preflight
rejects mixed aligner families.

| Aligner | Version used in TxNova's own testing | When |
|---------|--------------------------------------|------|
| **STAR** | 2.7.2a | Standard; splice-aware; pass a comprehensive GTF (`--sjdbGTFfile`) |
| **HISAT2** | any recent 2.x | Lower memory |

Pick one and use it for every sample in the sheet.


## 2. Build a genome index (once per reference)

**STAR:**

```bash
STAR --runMode genomeGenerate \
     --genomeDir star_index \
     --genomeFastaFiles genome.fa \
     --sjdbGTFfile annotation.gtf \
     --sjdbOverhang 99 \
     --runThreadN 8
```

Use the **same** `genome.fa` / `annotation.gtf` (or a name-and-length
compatible pair) you will later pass as `genome.fasta` /
`genome.annotation`. Preflight requires BAM `@SQ` names and lengths to
match the FASTA `.fai` exactly.

**HISAT2:**

```bash
hisat2-build genome.fa hisat2_index
```


## 3. Align each sample

**STAR** (paired-end; drop the second `--readFilesIn` file for single-end):

```bash
STAR --runMode alignReads \
     --genomeDir star_index \
     --readFilesIn sample_R1.fastq.gz sample_R2.fastq.gz \
     --readFilesCommand zcat \
     --outSAMtype BAM SortedByCoordinate \
     --runThreadN 8 \
     --outFileNamePrefix sample_
```

`--outSAMtype BAM SortedByCoordinate` sorts in one pass.

**HISAT2** (pipe into a coordinate-sorted BAM):

```bash
hisat2 -x hisat2_index -1 sample_R1.fastq.gz -2 sample_R2.fastq.gz -p 8 \
  | samtools sort -@ 8 -o sample_sorted.bam -
```


## 4. Sort, index, and verify

If the BAM is not already coordinate-sorted:

```bash
samtools sort -@ 8 -o sample_sorted.bam sample.bam
```

Every BAM needs a `.bai`:

```bash
samtools index sample_sorted.bam
```

Check contig names against the FASTA before you write `samples.tsv`:

```bash
samtools view -H sample_sorted.bam | grep '^@SQ' | head -3
grep '^>' genome.fa | head -3
```

Names must match literally (`chr1` vs `1` fails closed).


## 5. Figure out `strandedness`

`samples.tsv` needs one of `unstranded`, `fr`, or `rf` — **the same value
for every sample**. This is the kit, not something TxNova infers from the
BAM.

| Kit / protocol | `strandedness` |
|----------------|----------------|
| Illumina TruSeq Stranded (dUTP); most modern stranded kits | `rf` |
| Ligation-based stranded kits | `fr` |
| Unstranded (e.g. TruSeq RNA, no strand step) | `unstranded` |

If you do not know the kit, use
[RSeQC](http://rseqc.sourceforge.net/) `infer_experiment.py`:

```bash
infer_experiment.py -r annotation.bed12 -i sample_sorted.bam
```

- Mostly `"1++,1--,2+-,2-+"` (or `"++,--"` SE) → `fr`
- Mostly `"1+-,1-+,2++,2--"` (or `"+-,-+"` SE) → `rf`
- Roughly 50/50 → `unstranded`

A wrong value does not crash the run. It biases coverage and counts, and
shows up as odd valleys or empty funnels. Check this first if the tables
look implausible.


## 6. Mark duplicates (optional)

`quantify.skip_duplicate: auto` only drops `0x400` if preflight saw the
flag. To mark:

```bash
samtools markdup -@ 8 sample_sorted.bam sample_markdup.bam
samtools index sample_markdup.bam
```

Not required. On PCR-heavy libraries it changes the TPM gates.


## 7. Preflight on the bundled fixtures

The next cells use `tests/fixtures/` (tiny STAR BAMs + FASTA + GTF). They
do not need FASTQ or a genome index. This is the same check `txnova
preflight` runs on a real sheet.


In [2]:
def repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "tests" / "fixtures" / "ctrl_1.bam").is_file():
            return cand
    raise FileNotFoundError("tests/fixtures/ctrl_1.bam not found from " + str(here))


ROOT = repo_root()
FIX = ROOT / "tests" / "fixtures"
print(FIX)


/home/lieber/TxNova/tests/fixtures


In [3]:
sheet = pd.read_csv(FIX / "samples_ok.tsv", sep="\t")
sheet


,sample_id,bam,group,strandedness,replicate
0,ctrl_1,ctrl_1.bam,control,rf,1
1,ctrl_2,ctrl_2.bam,control,rf,2
2,treat_1,treat_1.bam,treat,rf,1
3,treat_2,treat_2.bam,treat,rf,2


A passing sheet: two control, two treat, one strandedness, BAM paths that
exist and are indexed.


In [4]:
import tempfile

raw = yaml.safe_load((FIX / "config_ok.yaml").read_text())
tmpdir = Path(tempfile.mkdtemp(prefix="txnova_prep_"))
raw["output_dir"] = str(tmpdir / "out")
raw["genome"]["fasta"] = str(FIX / "genome.fa")
raw["genome"]["annotation"] = str(FIX / "genes.gtf")
raw["samples"] = str(FIX / "samples_ok.tsv")
cfg_path = tmpdir / "config.yaml"
cfg_path.write_text(yaml.safe_dump(raw), encoding="utf-8")

cfg = load_config(cfg_path)
rows = load_samples(cfg.samples)
ok = run_preflight(cfg, rows)
print("ok", ok["ok"])
print("aligner_family", ok["aligner_family"])
print("library_layout", ok["library_layout"])
print("strandedness", ok.get("strandedness"))
print("n_control / n_treat", ok["n_control"], ok["n_treat"])
print("errors", ok.get("errors") or [])


ok True
aligner_family STAR
library_layout single
strandedness rf
n_control / n_treat 2 2
errors []


### The usual failure: `chr1` vs `1`

Same GTF intervals, seqnames without `chr`. Preflight fails closed.


In [5]:
raw_bad = yaml.safe_load(cfg_path.read_text())
raw_bad["genome"]["annotation"] = str(FIX / "genes_nochr.gtf")
bad_path = tmpdir / "config_nochr.yaml"
bad_path.write_text(yaml.safe_dump(raw_bad), encoding="utf-8")
cfg_bad = load_config(bad_path)
bad = run_preflight(cfg_bad, load_samples(cfg_bad.samples))
print("ok", bad["ok"])
for e in bad.get("errors") or []:
    print("-", e)


ok False
- GTF seqnames and BAM @SQ have empty intersection (GTF e.g. ["1"], BAM e.g. ["chr1"]). Names must match literally; TxNova will not rewrite chr prefixes.


### Mixed aligner family

A BAM whose `@PG` contains Bowtie2 is rejected even if STAR is also in
the header.


In [6]:
mix = tmpdir / "samples_bowtie.tsv"
mix.write_text(
    "sample_id\tbam\tgroup\tstrandedness\treplicate\n"
    f"ctrl_1\t{FIX / 'bowtie2.bam'}\tcontrol\trf\t1\n"
    f"ctrl_2\t{FIX / 'ctrl_2.bam'}\tcontrol\trf\t2\n"
    f"treat_1\t{FIX / 'treat_1.bam'}\ttreat\trf\t1\n"
    f"treat_2\t{FIX / 'treat_2.bam'}\ttreat\trf\t2\n",
    encoding="utf-8",
)
raw_mix = yaml.safe_load(cfg_path.read_text())
raw_mix["samples"] = str(mix)
mix_path = tmpdir / "config_bowtie.yaml"
mix_path.write_text(yaml.safe_dump(raw_mix), encoding="utf-8")
cfg_mix = load_config(mix_path)
mix_rep = run_preflight(cfg_mix, load_samples(cfg_mix.samples))
print("ok", mix_rep["ok"])
for e in mix_rep.get("errors") or []:
    print("-", e)


ok False
- sample ctrl_1: BAM @PG contains Bowtie2 (ID=bowtie2 PN=bowtie2); only STAR or HISAT2 are accepted. Bowtie2/minimap2 fail even when STAR is also in @PG.


## Sample sheet for a real run

```text
sample_id	bam	group	strandedness	replicate
ctrl_1	/data/ctrl_1_sorted.bam	control	rf	1
ctrl_2	/data/ctrl_2_sorted.bam	control	rf	2
treat_1	/data/treat_1_sorted.bam	treat	rf	1
treat_2	/data/treat_2_sorted.bam	treat	rf	2
```

```bash
txnova preflight -c config.yaml
```

That is the same check as §7. Next: {doc}`../quickstart`. Error strings:
{doc}`../faq`.
